In [ ]:
import torch
from torchvision.datasets.mnist import MNIST
import torchvision.transforms as transforms
import numpy as np
import random
import os
import pandas as pd
import torch.nn as nn
from collections import defaultdict
import seaborn as sns


# from GraphRicciCurvature.OllivierRicci import OllivierRicci
import networkx as nx
import community as community_louvain
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.integrate import simps
import pickle
import time
import pandas as pd
import sys
sys.path.append("..")

import tools.utils as utils
from tools.small_model import FC_MD
from RicciCurvature.OllivierRicci import OllivierRicci
from tools.FC_linear import FC_Linear
from tools.graph_curvature import graph_curvature_main_torch
from tools.draw_net import DrawNN


np.set_printoptions(threshold=np.inf)
torch.set_printoptions(threshold=torch.inf)

import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [ ]:
seed = 59
    
# set random seed
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# os.environ['CUDA_VISIBLE_DEVICES'] = '1' 
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
data_train = MNIST('./data/mnist',
                  train=True,
                  download=True,
                  transform=transforms.Compose([
                      # transforms.Resize((32, 32)),
                      transforms.ToTensor()]))

data_test = MNIST('./data/mnist',
                  train=False,
                  download=True,
                  transform=transforms.Compose([
                      # transforms.Resize((32, 32)),
                      transforms.ToTensor()]))




layers = [2, 4, 5, 6, 7]

model_zoo = {
    2: [784, 20, 15, 10],
    21: [784, 200, 150, 10],
    4: [784, 15, 25, 20, 15, 10],
    5: [784, 20, 30, 30, 20, 15, 10],
    6: [784, 20, 30, 30, 35, 20, 15, 10],
    7: [784, 30, 30, 40, 50, 30, 25, 20, 10]
}

selected_classes = [0,1,2,3,4,5,6,7,8,9]

In [ ]:
def standard_PGD(model, images, labels, device, eps=11/255, alpha=2/255, iters=40):
    images = images.to(device)
    labels = labels.to(device)
    loss = nn.CrossEntropyLoss()
        
    ori_images = images.data
        
    for i in range(iters) :    
        images.requires_grad = True
        outputs = model(images)

        model.zero_grad()
        cost = loss(outputs, labels).to(device)
        cost.backward()

        adv_images = images + alpha*images.grad.sign()
        eta = torch.clamp(adv_images - ori_images, min=-eps, max=eps)
        images = torch.clamp(ori_images + eta, min=0, max=1).detach_()
            
    return images


def test(n, loader, eps, alpha, iters, device):    
    n.eval()
    robust_pair = defaultdict(list)
    succ_pair = defaultdict(list)
 
    
    for l in selected_classes:
        for i, (images, labels) in enumerate(loader[l]):
            images = images.to(device)
            labels = labels.to(device)
            output = n(images)
            pred = output.detach().max(1)[1]
            
            adv_img = standard_PGD(n, images, labels, device, eps, alpha, iters)
            adv_out = n(adv_img)
            adv_pred = adv_out.detach().max(1)[1]
 
            robust_l = pred.eq(labels.view_as(pred)) & adv_pred.eq(labels.view_as(adv_pred))
            succ_l = pred.eq(labels.view_as(pred)) & ~adv_pred.eq(labels.view_as(adv_pred))

            succ_pair[l].append((images[succ_l].cpu(), adv_img[succ_l].cpu()))
            robust_pair[l].append((images[robust_l].cpu(), adv_img[robust_l].cpu()))

        # print(f'Finish label {l}....')

    return succ_pair, robust_pair

def get_fraction(curvature, b, dims):
    c = []
    layer_num = len(dims) - 1
    neg = np.zeros((layer_num), dtype=np.float32)
    top_neg = np.zeros((layer_num), dtype=np.float32)
    total_e = np.zeros((layer_num), dtype=np.float32)
    # neg = 0.
    # total_e = 0.
    
    for batch in range(b):
        ricci_curv = np.array(curvature[batch])
        for (i, j, curr) in ricci_curv:
            if curr > 1:
                continue
    
            l = int(i)
            if curr < 0:
                neg[l] += 1
            if curr < -10:
                top_neg[l] += 1
            total_e[l] += 1
            c.append(curr)
    return neg, total_e, top_neg, c


In [ ]:
# For FC
def layerwise_shortest_path_torch(dims, weights, device='cpu'):
    batch_size, edge_num = weights.shape
    num_layers = len(dims)
    paths = {}

    weight_idx = 0
    for i in range(num_layers - 1):
        src_size, dst_size = dims[i], dims[i+1]
        direct_dist = weights[:, weight_idx:weight_idx+src_size*dst_size]
        direct_dist = direct_dist.reshape(src_size, dst_size)
        # inf = torch.tensor(float('inf'), device=device)
        # paths[(i, i+1)] = np.where(direct_dist > 0, direct_dist, float('inf'))
        paths[(i, i+1)] = direct_dist
        weight_idx += src_size * dst_size

    return paths

In [ ]:
def vis_nodes(nodes, nodes_val):
    num_nodes = len(nodes[0])
    if num_nodes == 0:
        print("No nodes to visualize.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(12, 12))

    # **First Block (Default 28x28 if possible)**
    first_block_size = min(784, num_nodes)
    grid_size = int(np.sqrt(first_block_size))  # Adaptive for non-28x28 cases
    
    sns.heatmap(nodes[0][:first_block_size].reshape(grid_size, grid_size), cmap="coolwarm", ax=axes[0, 0])
    axes[0, 0].set_title(f"Ricci Curvature Nodes (First {grid_size}x{grid_size})")

    sns.heatmap(nodes_val[0][:first_block_size].reshape(grid_size, grid_size), cmap="coolwarm", ax=axes[0, 1])
    axes[0, 1].set_title(f"Original Nodes_Val (First {grid_size}x{grid_size})")

    # **Remaining Nodes (No Reshaping into Square)**
    if num_nodes > first_block_size:
        remaining_nodes = nodes[0][first_block_size:]
        remaining_nodes_val = nodes_val[0][first_block_size:]

        # If remaining nodes are too many, display them as a 1D heatmap.
        if len(remaining_nodes) > 100:
            sns.heatmap(remaining_nodes.reshape(1, -1), cmap="coolwarm", ax=axes[1, 0])
            axes[1, 0].set_title("Ricci Curvature Nodes (Remaining - 1D View)")

            sns.heatmap(remaining_nodes_val.reshape(1, -1), cmap="coolwarm", ax=axes[1, 1])
            axes[1, 1].set_title("Original Nodes_Val (Remaining - 1D View)")
        else:
            # If the remaining nodes fit nicely in 2D, just plot them as a 2D heatmap
            sns.heatmap(remaining_nodes.reshape(-1, 1), cmap="coolwarm", ax=axes[1, 0])
            axes[1, 0].set_title("Ricci Curvature Nodes (Remaining)")

            sns.heatmap(remaining_nodes_val.reshape(-1, 1), cmap="coolwarm", ax=axes[1, 1])
            axes[1, 1].set_title("Original Nodes_Val (Remaining)")

    plt.tight_layout()
    plt.show()



In [ ]:
def vis_edge(w1, w2, w3):
    # **Heatmap for each layer transition**
    for (layer_start, layer_end) in w1.keys():
        # if (layer_start == 0):
        #     continue
        
        fig, axes2 = plt.subplots(1, 2, figsize=(18, 6))

        # Edge Presence Heatmap
        # sns.heatmap(edge_p[(layer_start, layer_end)], cmap="coolwarm", ax=axes2[0])
        # axes2[0].set_title(f"Weight Value: Layer {layer_start} → {layer_end}")

        # Weight Heatmap
        sns.heatmap(w1[(layer_start, layer_end)], cmap="coolwarm", ax=axes2[0])
        axes2[0].set_title(f"Ori: Layer {layer_start} → {layer_end}")

        # Ricci Curvature Heatmap
        # print(f'Node value for layer {layer_start} - {layer_end} is {weight_p[(layer_start, layer_end)]}')
        sns.heatmap(w2[(layer_start, layer_end)], cmap="coolwarm", ax=axes2[1])
        axes2[1].set_title(f"Decay: Layer {layer_start} → {layer_end}")
        
        # sns.heatmap(w3[(layer_start, layer_end)], cmap="coolwarm", ax=axes2[2])
        # axes2[2].set_title(f"ADV: Layer {layer_start} → {layer_end}")

        plt.show()

In [ ]:

def visualization(dims, w1, w2, w3, b = 1):
    prefix_dims = np.cumsum([0] + dims).tolist()
    
    # weight_1 = layerwise_shortest_path_torch(dims, w1)
    # weight_2 = layerwise_shortest_path_torch(dims, w2)
    # weight_3 = layerwise_shortest_path_torch(dims, w3)
    
    # edges
    # vis_edge(weight_1, weight_2, None)
    vis_nodes(w1, w2)
    
    

In [ ]:
def get_top_c(curvature, b, dims, threshold = -50):
    c = []
    edge_set = set()
    edges = set()
    remaining_e = set()
    pos_e = set()
    
    for batch in range(b):
        ricci_curv = np.array(curvature[batch])
        for (i, j, curr) in ricci_curv:
            c.append((i,j,curr))
            if (curr < threshold):
                edge_set.add((i,j,curr))
                edges.add((i,j))
            elif (curr > 0):
                pos_e.add((i,j))
            elif (j > 818):
                remaining_e.add((i,j))

    c.sort(key=lambda x: x[2])
        
    return c, edge_set, edges, remaining_e, pos_e



def get_weights(curv_list, sp_dict, prefix_dims, ff):
    for (i,j,curv) in curv_list:
        i_layer = np.searchsorted(prefix_dims, i, side='right') - 1
        j_layer = np.searchsorted(prefix_dims, j, side='right') - 1
        i_idx = i - prefix_dims[i_layer]
        j_idx = j - prefix_dims[j_layer]
        sp = sp_dict[(i_layer, j_layer)][0, i_idx, j_idx]
        
        ff.write(f'For edge {i} - {j}: the weight is {sp}\n')
        if (i_layer > 0):
            pre_sp  = sp_dict[(i_layer-1, i_layer)]
            in_neigh = []
            for n in range(len(pre_sp[0,:,i_idx])):
                if pre_sp[0,n,i_idx] != float('inf'):
                    in_neigh.append(n)
            
            mid_l = set()
            for n in in_neigh:
                ff.write(f'For ingoing neighbor {n}: \n')
                for o in range(prefix_dims[j_layer] - prefix_dims[i_layer]):
                    if (pre_sp[0,n,o] != float('inf')):
                        mid_l.add(o)
                        ff.write(f'{n} - {o}: {pre_sp[0,n,o]}\n')
                ff.write(f'\n')



def save_shortest_paths_to_excel(shortest_paths, output_dir="shortest_paths_excel"):
    os.makedirs(output_dir, exist_ok=True)

    # Define the Excel file name
    output_file = f"{output_dir}/shortest_paths.xlsx"
    
    # Create an Excel writer
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        for (start, end), numpy_array in shortest_paths.items():
            # Loop through each 2D slice (batch) of the 3D NumPy array
            for i in range(numpy_array.shape[0]):
                df = pd.DataFrame(numpy_array[i].astype(np.float32))
                df.columns = [f'Node_{j}' for j in range(df.shape[1])]
                df.index = [f'Node_{j}' for j in range(df.shape[0])]
                # if (start == 1 and end == 2):
                #     print(df)
                #     print('='*150)
                # Save each batch as a separate sheet with a unique name
                sheet_name = f'shortest_path_{start}_to_{end}_Batch_{i+1}'
                df.to_excel(writer, sheet_name=sheet_name, index=True, index_label='Source Node')

            

In [ ]:
train_loader, test_loader, valid_loader, valid_dataset, test_dataset = utils.get_new_data(selected_classes, data_train, data_test, test_bs=2000, valid_num=5000)

sep_dataloader = utils.sep_label(test_dataset, selected_classes, bs=2000)

eps = [0.03, 0.07, 0.1, 0.2]
Q = [1]
# eps = [0.1]

model_type = 'fc'
model_pre_name = 'ori'
metric = 'q_inv'
res_path = 'res/' + metric + '/'
model_path = 'pgd/models/'
dataset = 'mnist'
alpha = 0
sample_num = 1

In [ ]:
model_full_n = model_type.lower() + model_pre_name.lower()
sample_size = sample_num

if not os.path.exists(res_path):
    os.makedirs(res_path)
    
layers = [2]
if 'big' in model_pre_name.lower():
    layers = [2]
    
dims = model_zoo[4]
    
model_name1 = "best_ori_10l_2.pth"
model_name2 = "decay/best_ori_10l_2.pth"
model_name3 = "pgdtrain_2.pth"

net_H1 = FC_MD(dims, 2)
net_H1.load_state_dict(torch.load(model_path + model_name1))
net_H1 = net_H1.to(device)

net_H2 = FC_MD(dims, 2)
net_H2.load_state_dict(torch.load(model_path + model_name2))
net_H2 = net_H2.to(device)

net_H3 = FC_MD(dims, 2)
net_H3.load_state_dict(torch.load(model_path + model_name3))
net_H3 = net_H3.to(device)

prefix_dims = np.cumsum([0] + dims).tolist()

neural_list = []
nodes_num = 0
edges_num = 0
i = 0
for p in net_H1.parameters():
    if i == 0:
        nodes_num += p.shape[1]
    if i%2 == 0:
        nodes_num += p.shape[0]
        edges_num += (p.shape[0] * p.shape[1])
        neural_list.append(p.shape[0])
    i += 1

# build model
with open("./curv_stats_2_all_label.txt", "w+") as ff:
    for e in eps:
        print(f'Current eps {e}: ')
        ff.write(f'Current eps {e}: \n')
        
        succ_pair1, robust_pair1 = test(net_H1, sep_dataloader, eps=e, alpha=2/255, iters=40, device=device)
        succ_pair2, robust_pair2 = test(net_H2, sep_dataloader, eps=e, alpha=2/255, iters=40, device=device)
        succ_pair3, robust_pair3 = test(net_H3, sep_dataloader, eps=e, alpha=2/255, iters=40, device=device)
        
        robust_c = defaultdict(list)
        nonrobust_c = defaultdict(list)
        non_fraction = defaultdict(list)
        rob_fraction = defaultdict(list)
            
        for l in selected_classes:
            print(f'Current label {l}: \n')
            ff.write(f'Current label {l}: \n')

            print(f'Robust pair')
            ff.write(f'Robust example: \n')
            # robust images
            count = 0
            for (ori_im, adv_im) in robust_pair1[l]:
                if (count >= sample_size):
                    break
                for (im, im_a) in zip(ori_im, adv_im):
                    img = im.to(device)
                    edge_array1, nodes_ori1, output1, all_node1 = net_H1.NN_info_batch(img.unsqueeze(0))
 
                    weights1 = output1.detach().clone().to(device)                   
                    weights1[edge_array1 == 0] = 0.
                    
                    img_a = im_a.to(device)
                    edge_array1_a, nodes_ori1_a, output1_a, all_node1_a = net_H1.NN_info_batch(img_a.unsqueeze(0))
 
                    weights1_a = output1_a.detach().clone().to(device)                   
                    weights1_a[edge_array1_a == 0] = 0.

                    weights_inv1 = net_H1.normalization_weight_w2(nodes_ori1, weights1, dims)
                    weights_inv1 = weights_inv1.detach()
                    ricci_curvature1, sp_dict1 = graph_curvature_main_torch(dims, weights_inv1, device=device, alpha=alpha)
                    weights_inv1_a = net_H1.normalization_weight_w2(nodes_ori1_a, weights1_a, dims)
                    weights_inv1_a = weights_inv1_a.detach()
                    ricci_curvature1_a, sp_dict1_a = graph_curvature_main_torch(dims, weights_inv1_a, device=device, alpha=alpha)
                    
                    e1 = nodes_ori1_a - nodes_ori1
                    
                    # ricci_curvature1, sp_dict1 = graph_curvature_main_torch(dims, weights_inv1, device=device, alpha=alpha)
                    
                    # c1, edge_set1, edges1, remaining_e1, pos_e1 = get_top_c(ricci_curvature1, 1, dims, threshold = -30)

                    # ff.write(f'For CE model {count}, the lowest curvature edges are {c1[0][0], c1[0][1], c1[0][2]} ({c1[0][0]-784, c1[0][1]-799, c1[0][2]}): \n')
                    # for edge in edge_set1:
                    #     ff.write(f'{edge[0], edge[1], edge[2]}\n')
                    
                    # ff.write("\n\n")
                    
                    # print('CE')
                    # save_shortest_paths_to_excel(sp_dict1, "./pandas/CE/sp_" + str(e) +'_' + str(l) + "_" + str(count))
                        
                    # name = f'class_{l}_{i}'
                    # network = DrawNN(dims, edges1, neural_list, pos_e1, remaining_e1)
                    # network.draw(name, '/')
                    
                    count += 1
                    if (count % 10 == 0):
                        print(f'Finish {count} graphs....')
                        
                    if (count >= sample_size):
                        break
                    
            count = 0
            for (ori_im, adv_im) in succ_pair1[l]:
                if (count >= sample_size):
                    break
                for (im, im_a) in zip(ori_im, adv_im):
                    img = im.to(device)
                    edge_array1, nodes_ori1, output1, all_node1 = net_H1.NN_info_batch(img.unsqueeze(0))
 
                    weights1 = output1.detach().clone().to(device)                   
                    weights1[edge_array1 == 0] = 0.
                    
                    img_a = im_a.to(device)
                    edge_array1_a, nodes_ori1_a, output1_a, all_node1_a = net_H1.NN_info_batch(img_a.unsqueeze(0))
 
                    weights1_a = output1_a.detach().clone().to(device)                   
                    weights1_a[edge_array1_a == 0] = 0.

                    weights_inv1 = net_H1.normalization_weight_w2(nodes_ori1, weights1, dims)
                    weights_inv1 = weights_inv1.detach()
                    weights_inv1_a = net_H1.normalization_weight_w2(nodes_ori1_a, weights1_a, dims)
                    weights_inv1_a = weights_inv1_a.detach()
                    
                    e1_non = nodes_ori1_a - nodes_ori1
                    
                    # ricci_curvature1, sp_dict1 = graph_curvature_main_torch(dims, weights_inv1, device=device, alpha=alpha)
                    
                    # c1, edge_set1, edges1, remaining_e1, pos_e1 = get_top_c(ricci_curvature1, 1, dims, threshold = -30)

                    # ff.write(f'For CE model {count}, the lowest curvature edges are {c1[0][0], c1[0][1], c1[0][2]} ({c1[0][0]-784, c1[0][1]-799, c1[0][2]}): \n')
                    # for edge in edge_set1:
                    #     ff.write(f'{edge[0], edge[1], edge[2]}\n')
                    
                    # ff.write("\n\n")
                    
                    # print('CE')
                    # save_shortest_paths_to_excel(sp_dict1, "./pandas/CE/sp_" + str(e) +'_' + str(l) + "_" + str(count))
                        
                    # name = f'class_{l}_{i}'
                    # network = DrawNN(dims, edges1, neural_list, pos_e1, remaining_e1)
                    # network.draw(name, '/')
                    
                    count += 1
                    if (count % 10 == 0):
                        print(f'Finish {count} graphs....')
                        
                    if (count >= sample_size):
                        break
                    
            count = 0
            for (ori_im, adv_im) in robust_pair2[l]:
                if (count >= sample_size):
                    break
                for (im, im_a) in zip(ori_im, adv_im):
                    img = im.to(device)
                    edge_array2, nodes_ori2, output2, all_node2 = net_H2.NN_info_batch(img.unsqueeze(0))
                    
                    img_a = im_a.to(device)
                    edge_array2_a, nodes_ori2_a, output2_a, all_node2_a = net_H1.NN_info_batch(img_a.unsqueeze(0))
 
                    weights2 = output2.detach().clone().to(device)                   
                    weights2[edge_array2 == 0] = 0.
       
                    weights_inv2 = net_H2.normalization_weight_w2(nodes_ori2, weights2, dims)
                    weights_inv2 = weights_inv2.detach()
                    
                    weights2_a = output2_a.detach().clone().to(device)                   
                    weights2_a[edge_array2_a == 0] = 0.
       
                    weights_inv2_a = net_H2.normalization_weight_w2(nodes_ori2_a, weights2_a, dims)
                    weights_inv2_a = weights_inv2_a.detach()
                    
                    e2 = nodes_ori2_a - nodes_ori2
                    
                    # ricci_curvature2, sp_dict2 = graph_curvature_main_torch(dims, weights_inv2, device=device, alpha=alpha)
                    
                    # c2, edge_set2, edges2, remaining_e2, pos_e2 = get_top_c(ricci_curvature2, 1, dims, threshold = -50)
                    # ff.write(f'For Decay model {count}, the lowest curvature edges are {c2[0][0], c2[0][1], c2[0][2]} ({c2[0][0]-784, c2[0][1]-799, c2[0][2]}): \n')
                    # for edge in edge_set2:
                    #     ff.write(f'{edge[0], edge[1], edge[2]}\n')
                        
                    # ff.write("\n\n")
                    
                    # print('DECAY')
                    # save_shortest_paths_to_excel(sp_dict2, "./pandas/Decay/sp_" + str(e) +'_' + str(l) + "_" + str(count))
                        
                    # name = f'class_{l}_{i}'
                    # network = DrawNN(dims, edges2, neural_list, pos_e2, remaining_e2)
                    # network.draw(name, '/')
         
                    count += 1
                    if (count % 10 == 0):
                        print(f'Finish {count} graphs....')
                        
                    if (count >= sample_size):
                        break
                    
            count = 0
            for (ori_im, adv_im) in succ_pair2[l]:
                if (count >= sample_size):
                    break
                for (im, im_a) in zip(ori_im, adv_im):
                    img = im.to(device)
                    edge_array2, nodes_ori2, output2, all_node2 = net_H2.NN_info_batch(img.unsqueeze(0))
                    
                    img_a = im_a.to(device)
                    edge_array2_a, nodes_ori2_a, output2_a, all_node2_a = net_H1.NN_info_batch(img_a.unsqueeze(0))
 
                    weights2 = output2.detach().clone().to(device)                   
                    weights2[edge_array2 == 0] = 0.
       
                    weights_inv2 = net_H2.normalization_weight_w2(nodes_ori2, weights2, dims)
                    weights_inv2 = weights_inv2.detach()
                    
                    weights2_a = output2_a.detach().clone().to(device)                   
                    weights2_a[edge_array2_a == 0] = 0.
       
                    weights_inv2_a = net_H2.normalization_weight_w2(nodes_ori2_a, weights2_a, dims)
                    weights_inv2_a = weights_inv2_a.detach()
                    
                    e2_non = nodes_ori2_a - nodes_ori2
                    
                    # ricci_curvature2, sp_dict2 = graph_curvature_main_torch(dims, weights_inv2, device=device, alpha=alpha)
                    
                    # c2, edge_set2, edges2, remaining_e2, pos_e2 = get_top_c(ricci_curvature2, 1, dims, threshold = -50)
                    # ff.write(f'For Decay model {count}, the lowest curvature edges are {c2[0][0], c2[0][1], c2[0][2]} ({c2[0][0]-784, c2[0][1]-799, c2[0][2]}): \n')
                    # for edge in edge_set2:
                    #     ff.write(f'{edge[0], edge[1], edge[2]}\n')
                        
                    # ff.write("\n\n")
                    
                    # print('DECAY')
                    # save_shortest_paths_to_excel(sp_dict2, "./pandas/Decay/sp_" + str(e) +'_' + str(l) + "_" + str(count))
                        
                    # name = f'class_{l}_{i}'
                    # network = DrawNN(dims, edges2, neural_list, pos_e2, remaining_e2)
                    # network.draw(name, '/')
         
                    count += 1
                    if (count % 10 == 0):
                        print(f'Finish {count} graphs....')
                        
                    if (count >= sample_size):
                        break
                    
            count = 0      
            for (ori_im, adv_im) in robust_pair3[l]:
                if (count >= sample_size):
                    break
                for (im, im_a) in zip(ori_im, adv_im):
                    img = im.to(device)
                    edge_array3, nodes_ori3, output3, all_node3 = net_H3.NN_info_batch(img.unsqueeze(0))
                    
                    img_a = im_a.to(device)
                    edge_array3_a, nodes_ori3_a, output3_a, all_node3_a = net_H1.NN_info_batch(img_a.unsqueeze(0))
 
                    weights3 = output3.detach().clone().to(device)                   
                    weights3[edge_array3 == 0] = 0.
       
                    weights_inv3 = net_H3.normalization_weight_w2(nodes_ori3, weights3, dims)
                    weights_inv3 = weights_inv3.detach()
                    
                    weights3_a = output3_a.detach().clone().to(device)                   
                    weights3_a[edge_array3_a == 0] = 0.
       
                    weights_inv3_a = net_H3.normalization_weight_w2(nodes_ori3_a, weights3_a, dims)
                    weights_inv3_a = weights_inv3_a.detach()
                    
                    e3 = nodes_ori3_a - nodes_ori3
                    # ricci_curvature3, sp_dict3 = graph_curvature_main_torch(dims, weights_inv3, device=device, alpha=alpha)

                    # c3, edge_set3, edges3, remaining_e3, pos_e3 = get_top_c(ricci_curvature3, 1, dims, threshold = -70)
                    # ff.write(f'For AT model {count}, the lowest curvature edges are {c3[0][0], c3[0][1], c3[0][2]} ({c3[0][0]-784, c3[0][1]-799, c3[0][2]}): \n')
                    # for edge in edge_set3:
                    #     ff.write(f'{edge[0], edge[1], edge[2]}\n')
                        
                    # ff.write("\n\n")
                    
                    # print('ADV')
                    # save_shortest_paths_to_excel(sp_dict3, "./pandas/AT/sp_" + str(e) +'_' + str(l) + "_" + str(count))
                        
                    # name = f'class_{l}_{i}'
                    # network = DrawNN(dims, edges3, neural_list, pos_e3, remaining_e3)
                    # network.draw(name, '/')
                    
                    count += 1
                    if (count % 10 == 0):
                        print(f'Finish {count} graphs....')
                        
                    if (count >= sample_size):
                        break
                    
            count = 0      
            for (ori_im, adv_im) in succ_pair3[l]:
                if (count >= sample_size):
                    break
                for (im, im_a) in zip(ori_im, adv_im):
                    img = im.to(device)
                    edge_array3, nodes_ori3, output3, all_node3 = net_H3.NN_info_batch(img.unsqueeze(0))
                    
                    img_a = im_a.to(device)
                    edge_array3_a, nodes_ori3_a, output3_a, all_node3_a = net_H1.NN_info_batch(img_a.unsqueeze(0))
 
                    weights3 = output3.detach().clone().to(device)                   
                    weights3[edge_array3 == 0] = 0.
       
                    weights_inv3 = net_H3.normalization_weight_w2(nodes_ori3, weights3, dims)
                    weights_inv3 = weights_inv3.detach()
                    
                    weights3_a = output3_a.detach().clone().to(device)                   
                    weights3_a[edge_array3_a == 0] = 0.
       
                    weights_inv3_a = net_H3.normalization_weight_w2(nodes_ori3_a, weights3_a, dims)
                    weights_inv3_a = weights_inv3_a.detach()
                    
                    e3_non = nodes_ori3_a - nodes_ori3
                    # ricci_curvature3, sp_dict3 = graph_curvature_main_torch(dims, weights_inv3, device=device, alpha=alpha)

                    # c3, edge_set3, edges3, remaining_e3, pos_e3 = get_top_c(ricci_curvature3, 1, dims, threshold = -70)
                    # ff.write(f'For AT model {count}, the lowest curvature edges are {c3[0][0], c3[0][1], c3[0][2]} ({c3[0][0]-784, c3[0][1]-799, c3[0][2]}): \n')
                    # for edge in edge_set3:
                    #     ff.write(f'{edge[0], edge[1], edge[2]}\n')
                        
                    # ff.write("\n\n")
                    
                    # print('ADV')
                    # save_shortest_paths_to_excel(sp_dict3, "./pandas/AT/sp_" + str(e) +'_' + str(l) + "_" + str(count))
                        
                    # name = f'class_{l}_{i}'
                    # network = DrawNN(dims, edges3, neural_list, pos_e3, remaining_e3)
                    # network.draw(name, '/')
                    
                    count += 1
                    if (count % 10 == 0):
                        print(f'Finish {count} graphs....')
                        
                    if (count >= sample_size):
                        break
                    
            # w_1 = weights_inv1_a - weights_inv1
            # w_2 = weights_inv2_a - weights_inv2
            # w_3 = weights_inv3_a - weights_inv3
                    
            # visualization(dims, e1.detach().cpu().numpy(), e1_non.detach().cpu().numpy(), w_3.detach().cpu().numpy())
            # visualization(dims, e1.detach().cpu().numpy(), e1_non.detach().cpu().numpy(), None)
            # visualization(dims, e2.detach().cpu().numpy(), e2_non.detach().cpu().numpy(), None)
            # visualization(dims, e3.detach().cpu().numpy(), e3_non.detach().cpu().numpy(), None)
